# 02b — FRED Resample Sensitivity: `.last()` vs `.mean()`

**Purpose.** Standalone diagnostic, not part of the ingest pipeline. Both
`02_fred_ingest_clean.ipynb` (`.mean()`) and `04_teammate_pandas_comparison.ipynb`
(`.last()`, matching the teammate) now key `MORTGAGE30US` to **month-end**, so the
two pipelines differ only in aggregation. This notebook quantifies how much that
one difference actually moves `mortgage_rate`, so the reconciliation flagged in
notebook 03's cell 12 has real numbers behind it before Phase 4 feature
engineering picks one.

Reads the same raw weekly file both ingest notebooks use
(`../data/raw/fred/mortgage30us.csv`) and writes nothing back to Postgres — read-only
comparison, no `mortgage_monthly` writes.


## 1. Setup

In [1]:
import os
import pandas as pd

pd.set_option("display.max_columns", 20)


## 2. Load weekly rates

Same raw file, same parsing as `02_fred_ingest_clean.ipynb` / `03_teammate_pandas_comparison.ipynb`.


In [2]:
RAW_PATH = "../data/raw/fred/mortgage30us.csv"
assert os.path.exists(RAW_PATH), f"Expected raw FRED file at {RAW_PATH} -- run 02_fred_ingest_clean.ipynb's download cell first."

weekly = pd.read_csv(RAW_PATH)
weekly.columns = ["date", "mortgage_rate"]
weekly["date"] = pd.to_datetime(weekly["date"])
weekly["mortgage_rate"] = pd.to_numeric(weekly["mortgage_rate"], errors="coerce")
weekly = weekly.dropna(subset=["mortgage_rate"]).sort_values("date").reset_index(drop=True)

print("Weekly rows:", len(weekly))
print("Date range:", weekly["date"].min().date(), "to", weekly["date"].max().date())
weekly.head()


Weekly rows: 2895
Date range: 1971-04-02 to 2026-09-17


,date,mortgage_rate
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29


## 3. Resample both ways, same month-end key

Both use `.resample("ME")` -- only the aggregation function changes. This isolates
the one variable notebook 03's cell 12 says the two pipelines actually differ on.


In [3]:
s = weekly.set_index("date")["mortgage_rate"]

monthly_last = s.resample("ME").last().rename("rate_last").to_frame()
monthly_mean = s.resample("ME").mean().rename("rate_mean").to_frame()

compare = monthly_last.join(monthly_mean, how="outer")
compare["diff"] = compare["rate_last"] - compare["rate_mean"]
compare["abs_diff"] = compare["diff"].abs()

compare.index.name = "month"
compare = compare.reset_index()
compare.head()


,month,rate_last,rate_mean,diff,abs_diff
0,1971-04-30,7.29,7.3100,-0.0200,0.0200
1,1971-05-31,7.46,7.4250,0.0350,0.0350
2,1971-06-30,7.54,7.5300,0.0100,0.0100
3,1971-07-31,7.69,7.6040,0.0860,0.0860
4,1971-08-31,7.69,7.6975,-0.0075,0.0075


## 4. How much does it actually move `mortgage_rate`?

Summary stats on `abs_diff`, plus the months where the two conventions disagree
most -- these are the months most likely to matter if the rate-of-change feature
(Phase 4) is sensitive to which aggregation was used.


In [4]:
n_months = len(compare)
print(f"Months compared: {n_months:,}")
print(f"Mean |diff|:   {compare['abs_diff'].mean():.4f} pp")
print(f"Median |diff|: {compare['abs_diff'].median():.4f} pp")
print(f"Max |diff|:    {compare['abs_diff'].max():.4f} pp")
print(f"Corr(rate_last, rate_mean): {compare['rate_last'].corr(compare['rate_mean']):.6f}")

# Months with the biggest disagreement -- typically fast-moving-rate months,
# where "last observed" and "average over the month" diverge most.
compare.reindex(compare["abs_diff"].sort_values(ascending=False).index).head(10)


Months compared: 666
Mean |diff|:   0.0800 pp
Median |diff|: 0.0550 pp
Max |diff|:    1.0620 pp
Corr(rate_last, rate_mean): 0.999247


,month,rate_last,rate_mean,diff,abs_diff
109,1980-05-31,13.20,14.2620,-1.0620,1.0620
107,1980-03-31,16.03,15.2825,0.7475,0.7475
127,1981-11-30,17.21,17.8250,-0.6150,0.6150
617,2022-09-30,6.70,6.1120,0.5880,0.5880
106,1980-02-29,13.59,13.0400,0.5500,0.5500
192,1987-04-30,10.37,9.8325,0.5375,0.5375
611,2022-03-31,4.67,4.1720,0.4980,0.4980
138,1982-10-31,14.15,14.6080,-0.4580,0.4580
121,1981-05-31,16.80,16.4020,0.3980,0.3980
112,1980-08-31,12.95,12.5600,0.3900,0.3900


## 5. Rate-of-change sensitivity

The feature that actually feeds the model (per the Team Reference doc's field
scope) is `MORTGAGE30US` rate-of-change, not the level itself -- so the more
relevant question is whether the two conventions still agree on month-over-month
*direction* and magnitude of change, not just on the level.


In [5]:
compare["chg_last"] = compare["rate_last"].diff()
compare["chg_mean"] = compare["rate_mean"].diff()
compare["chg_diff"] = compare["chg_last"] - compare["chg_mean"]

sign_mismatch = (
    compare.dropna(subset=["chg_last", "chg_mean"])
    .loc[lambda d: (d["chg_last"] > 0) != (d["chg_mean"] > 0)]
)

print(f"Months where month-over-month direction disagrees: {len(sign_mismatch)} / {compare['chg_last'].notna().sum()}")
print(f"Mean |chg_diff|: {compare['chg_diff'].abs().mean():.4f} pp")
sign_mismatch[["month", "chg_last", "chg_mean"]].head(10)


Months where month-over-month direction disagrees: 111 / 665
Mean |chg_diff|: 0.1059 pp


,month,chg_last,chg_mean
4,1971-08-31,0.00,0.0935
12,1972-04-30,0.10,-0.0080
32,1973-12-31,0.01,-0.0445
33,1974-01-31,-0.04,0.0025
42,1974-10-31,-0.09,0.0200
50,1975-06-30,0.01,-0.0135
78,1977-10-31,0.00,0.0265
79,1977-11-30,0.03,0.0000
94,1979-02-28,0.00,0.0175
99,1979-07-31,-0.02,0.0565


## 6. Takeaway


Looking at the executed results: across all 666 months, `.last()` and `.mean()` track each other very closely at the level, correlation of 0.9992, median absolute difference of only 0.055 percentage points. The biggest single-month gaps show up in the early-1980s Volcker-era rate spikes (May 1980 diverges by over 1 full point) and in 2022, when rates were rising fast. If the model only used the raw rate level, this difference would barely matter.

But the actual feature (per your Team Reference doc's field scope) is rate-of-change, not the level, and that's where the picture changes. The two methods disagree on the _direction_ of the month-over-month change in 111 of 665 months, about one in six, with an average magnitude gap of about 0.11 percentage points when they do disagree. And two of the ten largest level-divergence months are September and March 2022, sitting right inside the test window that's driving your project's elevated decline rate. 

So the takeaway is that this isn't a difference you can wave off: since the model reads rate-of-change rather than level, and the two conventions flip sign on that change in roughly 1 out of 6 months, concentrated in the exact period the target labels care about most, notebook 03's flagged reconciliation needs to become an actual decision rather than staying an open note. Picking `.mean()` to match `02_fred_ingest_clean.ipynb` (rather than the teammate's `.last()`) would keep this project consistent with its main ingest pipeline, but that's a call worth making deliberately before Phase 4 feature engineering starts, not by default.